In [1]:
# Project-relative paths (supports notebooks launched from the project root or notebooks/).
from pathlib import Path
import sys

_PROJECT_CANDIDATES = (Path.cwd(), *Path.cwd().parents)
PROJECT_ROOT = next(
    (p for p in _PROJECT_CANDIDATES if (p / "data" / "raw" / "mongo_test.test-result.json").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Launch Jupyter from the project root or notebooks/ directory.")
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "artifacts" / "models"
PREPROCESSING_DIR = PROJECT_ROOT / "artifacts" / "preprocessing"
SPLITS_DIR = PROJECT_ROOT / "artifacts" / "splits"
for _path in (DATA_PROCESSED, MODELS_DIR, PREPROCESSING_DIR, SPLITS_DIR):
    _path.mkdir(parents=True, exist_ok=True)
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))


In [2]:
import numpy as np
import pandas as pd
import joblib
from itertools import product
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_predict, KFold
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from kafka_preprocessing import TargetScaler  # needed to deserialize target_scaler.pkl

BASE_MODELS = ["Random Forest", "XGBoost", "GPR"]


In [3]:
# Base estimators used by the three-model Stacking ensemble.
rf_model  = joblib.load(MODELS_DIR / "rf_model.pkl")
xgb_model = joblib.load(MODELS_DIR / "xgb_model.pkl")
gpr_model = joblib.load(MODELS_DIR / "gpr_model.pkl")

# LightGBM is evaluated independently; it is NOT an input to the Ridge meta-model.
lgbm_model = joblib.load(MODELS_DIR / "lgbm_model.pkl")

X_train_s     = joblib.load(SPLITS_DIR / "X_train_s.pkl")
X_test_s      = joblib.load(SPLITS_DIR / "X_test_s.pkl")
y_train       = joblib.load(SPLITS_DIR / "y_train.pkl")
y_test        = joblib.load(SPLITS_DIR / "y_test.pkl")
y_train_s     = joblib.load(SPLITS_DIR / "y_train_s.pkl")
target_scaler = joblib.load(PREPROCESSING_DIR / "target_scaler.pkl")
preprocessor  = joblib.load(PREPROCESSING_DIR / "preprocessor.pkl")

assert len(X_train_s) == len(y_train) == len(y_train_s)
assert len(X_test_s) == len(y_test)
print("✅ مدل‌های پایه، داده‌ها و پیش‌پردازنده بارگذاری شدند.")


✅ مدل‌های پایه، داده‌ها و پیش‌پردازنده بارگذاری شدند.


In [4]:
# Generate out-of-fold (OOF) predictions only from the three selected base models.
# Keep the original five-fold CV protocol for comparability with the archived experiment.
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Random Forest is a MultiOutputRegressor (it expects a two-dimensional y).
pred_rf_train = np.asarray(
    cross_val_predict(rf_model, X_train_s, y_train, cv=kf)
).ravel()

# XGBoost expects a one-dimensional y in this project.
pred_xgb_train = np.asarray(
    cross_val_predict(xgb_model, X_train_s, y_train.to_numpy().ravel(), cv=kf)
).ravel()

# GPR was trained against a scaled throughput target: transform back to docs/sec.
pred_gpr_train_scaled = np.asarray(
    cross_val_predict(gpr_model, X_train_s, np.asarray(y_train_s).ravel(), cv=kf)
).reshape(-1, 1)
pred_gpr_train = target_scaler.scalers["throughput"].inverse_transform(
    pred_gpr_train_scaled
).ravel()

# Column order must match the test-set and recommendation matrices below.
X_meta_train = pd.DataFrame(
    np.column_stack([pred_rf_train, pred_xgb_train, pred_gpr_train]),
    columns=BASE_MODELS,
)
assert X_meta_train.shape == (len(X_train_s), 3)
print(f"✅ فراویژگی‌های سه‌مدلی (OOF) ساخته شدند: {X_meta_train.shape}")


✅ فراویژگی‌های سه‌مدلی (OOF) ساخته شدند: (998, 3)


## 2. Fit the three-input Ridge meta-regressor

Train a **new** Ridge model on three OOF features. The four-input Ridge coefficients from the legacy notebook are invalid here.

In [5]:
# Train a NEW meta-model. Old four-model coefficients cannot be reused.
meta_model = Ridge(alpha=1.0)
meta_model.fit(X_meta_train, y_train.to_numpy().ravel())

print("✅ Ridge سه‌مدلی آموزش داده شد.")
for name, coef in zip(BASE_MODELS, meta_model.coef_):
    print(f"  {name:<14}: {coef:.4f}")
print(f"  intercept     : {meta_model.intercept_:.4f}")


✅ Ridge سه‌مدلی آموزش داده شد.
  Random Forest : 0.5102
  XGBoost       : 0.5086
  GPR           : 0.0379
  intercept     : -5064.2441


In [6]:
# Final predictions of the fitted base models on the HELD-OUT test set.
pred_rf_test = np.asarray(rf_model.predict(X_test_s)).ravel()
pred_xgb_test = np.asarray(xgb_model.predict(X_test_s)).ravel()
pred_gpr_test = target_scaler.scalers["throughput"].inverse_transform(
    np.asarray(gpr_model.predict(X_test_s)).reshape(-1, 1)
).ravel()

# A standalone LightGBM benchmark, excluded from Stacking meta-features.
pred_lgbm_test = np.asarray(lgbm_model.predict(X_test_s)).ravel()

X_meta_test = pd.DataFrame(
    np.column_stack([pred_rf_test, pred_xgb_test, pred_gpr_test]),
    columns=BASE_MODELS,
)
assert X_meta_test.shape == (len(X_test_s), 3)
y_pred_stacking = meta_model.predict(X_meta_test)
true = y_test.to_numpy().ravel()

models = {
    "Random Forest": pred_rf_test,
    "XGBoost": pred_xgb_test,
    "GPR": pred_gpr_test,
    "LightGBM (standalone)": pred_lgbm_test,
    "Stacking (3 models)": y_pred_stacking,
}
print("\n" + "=" * 66)
print("Evaluation on saved test split — corrected 3-model stacking")
print("=" * 66)
print(f"{'Model':<24} {'R²':>8} {'MAE':>13} {'RMSE':>13}")
print("-" * 66)
for name, preds in models.items():
    r2 = r2_score(true, preds)
    mae = mean_absolute_error(true, preds)
    rmse = np.sqrt(mean_squared_error(true, preds))
    print(f"{name:<24} {r2:>8.4f} {mae:>13.2f} {rmse:>13.2f}")



Evaluation on saved test split — corrected 3-model stacking
Model                          R²           MAE          RMSE
------------------------------------------------------------------
Random Forest              0.8623       6662.61      10420.27
XGBoost                    0.8824       6584.96       9627.00
GPR                        0.8211       7565.13      11876.04
LightGBM (standalone)      0.8556       7314.17      10667.87
Stacking (3 models)        0.8851       6386.28       9518.63


D:\Amin\My Python Projects\MFT-Kafka-Ensemble-Optimization\.venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [7]:
# Restrict grid values to those represented in the project's experiment dataset.
BATCH_SIZE_VALUES  = [16384, 32768, 49152, 65536]
LINGER_MS_VALUES   = list(range(5, 506, 20))
COMPRESSION_VALUES = ["none", "snappy", "lz4", "gzip"]


def recommend_stacking(bill_size, top_n=3):
    """Recommend configurations using only RF, XGBoost and GPR + trained Ridge."""
    if not isinstance(top_n, int) or top_n < 1:
        raise ValueError("top_n must be a positive integer")

    combos = list(product(BATCH_SIZE_VALUES, LINGER_MS_VALUES, COMPRESSION_VALUES))
    grid = pd.DataFrame(combos, columns=["batchSize", "lingerMs", "compressionType"])
    grid["billSize"] = bill_size

    X_grid = preprocessor.transform(grid)
    p_rf = np.asarray(rf_model.predict(X_grid)).ravel()
    p_xgb = np.asarray(xgb_model.predict(X_grid)).ravel()
    p_gpr = target_scaler.scalers["throughput"].inverse_transform(
        np.asarray(gpr_model.predict(X_grid)).reshape(-1, 1)
    ).ravel()

    # LightGBM intentionally omitted; same column order as X_meta_train.
    X_meta_grid = pd.DataFrame(
        np.column_stack([p_rf, p_xgb, p_gpr]),
        columns=BASE_MODELS,
    )
    assert X_meta_grid.shape == (len(grid), 3)
    grid["predicted_throughput"] = meta_model.predict(X_meta_grid)
    grid = grid.sort_values("predicted_throughput", ascending=False)

    print(f"\n{'=' * 50}")
    print(f"Stacking (3 models) | billSize = {bill_size:,}")
    print(f"{'=' * 50}")
    for rank, (_, row) in enumerate(grid.head(top_n).iterrows(), 1):
        print(f"\nRank #{rank}")
        print(f"  batchSize       : {int(row['batchSize'])}")
        print(f"  lingerMs        : {int(row['lingerMs'])} ms")
        print(f"  compressionType : {row['compressionType']}")
        print(f"  throughput      : {row['predicted_throughput']:,.0f} docs/sec")

    return grid.head(top_n).reset_index(drop=True)


In [8]:
recommend_stacking(bill_size=309000, top_n=3)
recommend_stacking(bill_size=618000, top_n=3)
recommend_stacking(bill_size=927000, top_n=3)



Stacking (3 models) | billSize = 309,000

Rank #1
  batchSize       : 65536
  lingerMs        : 405 ms
  compressionType : snappy
  throughput      : 179,686 docs/sec

Rank #2
  batchSize       : 65536
  lingerMs        : 325 ms
  compressionType : lz4
  throughput      : 179,612 docs/sec

Rank #3
  batchSize       : 65536
  lingerMs        : 385 ms
  compressionType : lz4
  throughput      : 179,612 docs/sec

Stacking (3 models) | billSize = 618,000

Rank #1
  batchSize       : 65536
  lingerMs        : 445 ms
  compressionType : none
  throughput      : 163,253 docs/sec

Rank #2
  batchSize       : 65536
  lingerMs        : 385 ms
  compressionType : none
  throughput      : 162,905 docs/sec

Rank #3
  batchSize       : 65536
  lingerMs        : 405 ms
  compressionType : none
  throughput      : 162,348 docs/sec

Stacking (3 models) | billSize = 927,000

Rank #1
  batchSize       : 65536
  lingerMs        : 85 ms
  compressionType : none
  throughput      : 146,473 docs/sec

Rank #

,batchSize,lingerMs,compressionType,billSize,predicted_throughput
0,65536,85,none,927000,146472.785480
1,49152,85,none,927000,140966.315803
2,65536,65,none,927000,140387.513289
